| título | projeto | versão | data | autores | status |
| :--- | :--- | :--- | :--- | :--- | :--- |
| CRISP-DM — Fase 3: Data Preparation | Projeção da Taxa de Congestionamento — Justiça Estadual (GO) | 1.0 | 01-12-2025 | Júlio César e Lays de Freitas | Rascunho |

Esse Notebook contém o **pré-processamento dos dados**.

### BIBLIOTECAS

In [12]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import glob

from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from datetime import date

### CARREGAMENTO DOS DADOS

In [13]:
# Listar os arquivos CSV na pasta 'uploads'
arquivos_csv = glob.glob('uploads/processos_*.csv')

# Carregar os arquivos CSV e concatenar em um único DataFrame
dfs = []
for arquivo in arquivos_csv:
    
    df_temp = pd.read_csv(arquivo, sep=',', encoding='utf-8')
    dfs.append(df_temp)

dataset = pd.concat(dfs, ignore_index=True)

print("\n=== Arquivo carregado com sucesso! ===")
print("Dimensões (linhas, colunas):", dataset.shape)


=== Arquivo carregado com sucesso! ===
Dimensões (linhas, colunas): (3245632, 10)


### GRAVANDO UMA CÓPIA PARA TRABALHO

In [14]:
df = dataset.copy()

### AMOSTRA DOS DADOS

In [15]:
df.head()

,processo,data_distribuicao,data_baixa,entrancia,comarca,serventia,nome_area_acao,is_segredo_justica,codg_classe,codg_assuntos
0,0119071.75.2004.8.09.0051,2022-05-25,2022-06-30,FINAL,GOIÂNIA,2ª Vara Cível,upj civel,False,7.0,10671
1,0168391.94.2004.8.09.0051,2022-05-20,2022-05-20,FINAL,GOIÂNIA,3ª Vara Cível,upj civel,False,7.0,10671
2,0189657.40.2004.8.09.0051,2022-06-02,2024-01-22,FINAL,GOIÂNIA,31ª Vara Cível,upj civel,False,7.0,10671
3,0197944.89.2004.8.09.0051,2022-06-07,2022-10-07,FINAL,GOIÂNIA,22ª Vara Cível,upj civel,False,7.0,10671
4,0211274.56.2004.8.09.0051,2022-06-09,2022-08-03,FINAL,GOIÂNIA,8ª Vara Cível,upj civel,False,7.0,10671


### LIMPEZA E TRATAMENTO DOS DADOS

In [16]:
# Verificar o nome correto das colunas (pode haver diferenças de acentuação ou espaços)
colunas = df.columns.tolist()

# Encontrar as colunas de data corretamente
coluna_serventia = [col for col in colunas if 'serventia' in col.lower()][0]
coluna_distribuicao = [col for col in colunas if 'data_distribuicao' in col.lower()][0]
coluna_baixa = [col for col in colunas if 'data_baixa' in col.lower()][0]
coluna_area_acao = [col for col in colunas if 'nome_area_acao' in col.lower()][0]
coluna_processo_id = [col for col in colunas if 'processo' in col.lower()][0]
coluna_comarca = [col for col in colunas if 'comarca' in col.lower()][0]

# Renomear colunas para garantir consistência
df = df.rename(columns={
coluna_distribuicao: 'data_distribuicao',
coluna_baixa: 'data_baixa',
coluna_area_acao: 'nome_area_acao',
coluna_processo_id: 'processo',
coluna_comarca: 'comarca',
coluna_serventia: 'serventia'
})

# Converter colunas de data para datetime com tratamento de erros
df['data_distribuicao'] = pd.to_datetime(df['data_distribuicao'], errors='coerce')
df['data_baixa'] = pd.to_datetime(df['data_baixa'], errors='coerce')

### CONSTRUÇÃO DO DATAFRAME DE TREINO E TESTE

In [ ]:
# CRIAÇÃO DAS ESTATÍSTICAS POR MÊS ('comarca' e 'serventia'):

# 1. Extrair MÊS e ANO das colunas data_distribuicao e data_baixa
df['mes_distribuicao'] = df['data_distribuicao'].dt.month
df['mes_baixa'] = df['data_baixa'].dt.month
df['ano_distribuicao'] = df['data_distribuicao'].dt.year
df['ano_baixa'] = df['data_baixa'].dt.year

# Chave de agrupamento para as estatísticas
grouping_keys = ['comarca', 'serventia']

# 2. Cálculos MENSAIS
# 2.1 Distribuídos por MÊS
distribuidos_mes_df = df.dropna(subset=['ano_distribuicao', 'mes_distribuicao']).groupby(
    ['ano_distribuicao', 'mes_distribuicao'] + grouping_keys
).size().reset_index(name='Distribuídos_mes')
distribuidos_mes_df = distribuidos_mes_df.rename(columns={'ano_distribuicao': 'ano', 'mes_distribuicao': 'mes'})

# 2.2 Baixados por MÊS
baixados_mes_df = df.dropna(subset=['ano_baixa', 'mes_baixa']).groupby(
    ['ano_baixa', 'mes_baixa'] + grouping_keys
).size().reset_index(name='Baixados_mes')
baixados_mes_df = baixados_mes_df.rename(columns={'ano_baixa': 'ano', 'mes_baixa': 'mes'})

# 2.3 Pendentes por MÊS
pendentes_mes_df = df[df['data_baixa'].isna()].dropna(subset=['ano_distribuicao', 'mes_distribuicao']).groupby(
    ['ano_distribuicao', 'mes_distribuicao'] + grouping_keys
).size().reset_index(name='Pendentes_mes')
pendentes_mes_df = pendentes_mes_df.rename(columns={'ano_distribuicao': 'ano', 'mes_distribuicao': 'mes'})

# 4. Junção e Limpeza (MENSAL)
merge_keys_mes = ['ano', 'mes'] + grouping_keys
estatisticas_mes = pd.merge(distribuidos_mes_df, baixados_mes_df, on=merge_keys_mes, how='outer')
estatisticas_mes = pd.merge(estatisticas_mes, pendentes_mes_df, on=merge_keys_mes, how='outer')
estatisticas_mes = estatisticas_mes.fillna(0)
estatisticas_mes[['Distribuídos_mes', 'Baixados_mes', 'Pendentes_mes']] = estatisticas_mes[['Distribuídos_mes', 'Baixados_mes', 'Pendentes_mes']].astype(int)
estatisticas_mes = estatisticas_mes.dropna(subset=['ano', 'mes'])
estatisticas_mes[['ano', 'mes']] = estatisticas_mes[['ano', 'mes']].astype(int)

# 5. Cálculo da Taxa de Congestionamento (MENSAL)
soma_mensal = estatisticas_mes['Pendentes_mes'] + estatisticas_mes['Baixados_mes']
estatisticas_mes['Taxa de Congestionamento_mes (%)'] = np.where(
    soma_mensal > 0, (estatisticas_mes['Pendentes_mes'] / soma_mensal) * 100, 0
).round(2)

# 6. Montando o dataframe
estatisticas_mes = estatisticas_mes.sort_values(by=['ano', 'mes', 'comarca', 'serventia'], ascending=[False, True, True, True])
colunas_finais_mes = [
    'ano', 'mes', 'comarca', 'serventia',
    'Distribuídos_mes', 'Baixados_mes', 'Pendentes_mes', 'Taxa de Congestionamento_mes (%)'
]

df_estatisticas_mes = estatisticas_mes[colunas_finais_mes]

# 7. Amostra do dataframe tratado
df_estatisticas_mes.head()


,ano,mes,comarca,serventia,Distribuídos_mes,Baixados_mes,Pendentes_mes,Taxa de Congestionamento_mes (%)
19253,2025,1,ABADIÂNIA,Vara Judicial,140,66,80,54.79
19254,2025,1,ACREÚNA,"1ª Vara Judicial (Família e Sucessões, Infânci...",109,109,43,28.29
19255,2025,1,ACREÚNA,"2ª Vara Judicial (Fazendas Públicas, Criminal,...",67,84,41,32.80
19256,2025,1,ALEXÂNIA,Vara Judicial,258,225,99,30.56
19257,2025,1,ALTO PARAÍSO DE GOIÁS,Vara Judicial,155,70,96,57.83


### SEPARAR CONJUNTOS DE TREINO (80%) E TESTE (20%) 

In [25]:
train, test_split = train_test_split(df_estatisticas_mes.copy(), test_size=0.2)

### AMOSTRA DO CONJUNTO TREINO

In [21]:
train.head()

,ano,mes,comarca,serventia,Distribuídos_mes,Baixados_mes,Pendentes_mes,Taxa de Congestionamento_mes (%)
10489,2023,9,ANÁPOLIS,4º Juizado Especial Cível,306,288,12,4.00
22250,2025,6,GOIÂNIA,4º Juízo do Núcleo de Justiça 4.0 especializad...,619,559,522,48.29
23513,2025,8,RIO VERDE,3ª Vara Criminal (crimes em geral),121,143,69,32.55
14968,2024,5,FLORES DE GOIÁS,Vara Judicial,48,36,19,34.55
2719,2022,6,GOIANIRA,Juizado Especial Cível e Criminal,120,20,1,4.76


### GRAVAR O CONJUNTO TREINO PRÉ-PROCESSADO

In [22]:
train.to_csv('datasets/train-processed.csv', index=False)

### AMOSTRA DO CONJUNTO TESTE

In [26]:
test_split.head()

,ano,mes,comarca,serventia,Distribuídos_mes,Baixados_mes,Pendentes_mes,Taxa de Congestionamento_mes (%)
16217,2024,7,GUAPÓ,"2ª Vara Judicial (Fazendas Públicas, Criminal,...",135,151,33,17.93
18063,2024,10,TRIBUNAL DE JUSTIÇA,GABINETE DES. GERSON SANTANA CINTRA,93,61,2,3.17
4010,2022,8,PORANGATU,15º CEJUSC REGIONAL VIRTUAL DO INTERIOR,118,119,0,0.00
16434,2024,7,TRIBUNAL DE JUSTIÇA,GABINETE DES. SILVÂNIO DIVINO DE ALVARENGA,99,102,0,0.00
20403,2025,3,APARECIDA DE GOIÂNIA,2º Juizado de Violência Doméstica e Familiar c...,196,137,61,30.81


### GRAVAR CONJUNTO TESTE PRÉ-PROCESSADO

In [ ]:
test_split.to_csv('datasets/test_split.csv', index=False)